# MGS-07e : CMA-ES natif — ce que le rang-un achète sur une vallée orientée

[← MGS-7d MichalewiczDixonPrice](MGS-07d-MichalewiczDixonPrice.ipynb) · [MGS-8 LandscapeExplorer →](MGS-08-LandscapeExplorer.ipynb)

**Objectifs** : voir fonctionner les trois mécanismes que CMA-ES ajoute au rang-μ déjà porté par xNES — la mise à jour de **rang un** (chemin de covariance $p_c$), l'**adaptation du pas par cumul de trajectoire** (CSA, $p_\sigma$) et le **redémarrage sur stagnation** — puis lire dans la covariance apprise l'**orientation de la vallée**, la grandeur que MGS-07d mesurait à l'œil sur ses heatmaps. **Prérequis** : MGS-07b à 07d (les paysages projetés), Search-05b (xNES/SNES et la référence Python `cma`). **Durée** : ~40 min.

## Préparation

Depuis la racine du dépôt, initialiser le sous-module et construire la dépendance exacte avant d'ouvrir ce carnet :

```bash
git submodule update --init --recursive MyIA.AI.Notebooks/Search/MetaGeneticSharp
dotnet build MyIA.AI.Notebooks/Search/MetaGeneticSharp/src/MetaGeneticSharp.Domain/MetaGeneticSharp.Domain.csproj -c Release
```

.NET 9 et le noyau .NET Interactive sont requis. Le gitlink fixe MetaGeneticSharp à `aba0a6233ab1cfa6f5e11289cc649231aef0d1ed` et son sous-module GeneticSharp ; aucune clé ni GPU n'est nécessaire. Exécuter avec le répertoire courant de ce carnet. Les graphiques sont des SVG statiques : seules les DLL du projet `Domain` (Release) sont référencées, ni SkiaSharp ni le projet `Extensions`.

## Ce que ce carnet teste

**Question pédagogique** : sur un ellipsoïde tourné à fort conditionnement — le paysage qui a servi à valider l'implémentation — que gagne concrètement le chemin de rang un sur la seule recombinaison pondérée du rang-μ (xNES) ? Et où se lit, dans l'état de la distribution, l'apprentissage de la géométrie ?

**Hypothèses à vérifier** :

- Les méthodes du banc (CMA-ES, son ablation, xNES, SNES, DE) partagent le même canevas GA, la même population et le même budget : les écarts mesurés viennent des mécanismes, pas du câblage.
- SNES (covariance diagonale) souffre sur une vallée **tournée** — l'axe croisé qu'il ne peut pas représenter est précisément l'orientation ; DE, sans modèle de distribution, s'écrase ; xNES tient ; CMA-ES atteint la cible $10^{-8}$ en moins d'évaluations que xNES.
- **Le rang-un est la cause de l'écart, pas un accompagnement.** Le cinquième bras est le **même** CMA-ES avec le seul coefficient $c_1$ neutralisé — CSA, politique de redémarrage, budget et initialisation inchangés —, comparé graine par graine au bras complet, xNES restant la référence externe. Un effet faible ou nul de cette ablation réfuterait la lecture « le rang-un achète les ordres de grandeur » : c'est la mesure qui tranche, pas la prose.
- En 2-D, le rapport des valeurs propres et l'angle de l'axe principal de la covariance convergent vers les valeurs géométriques de la vallée — la réponse algébrique à l'exercice 3 de MGS-07d (gradient vertical/horizontal).

**Plafond déclaré, pas contourné** : le redémarrage implémenté est *restart-on-stall* — covariance réinitialisée autour du meilleur point, **population inchangée**. L'IPOP propre (population doublée à chaque redémarrage) n'est pas accessible depuis un composé ; la section dédiée cite la déclaration de la classe et l'exercice 1 construit la boucle côté appelant, comme elle le prescrit.

## Reproductibilité

Graine `4242` pour la passe de référence (celle du test unitaire), graines {4242, 1717, 2718} pour le banc. Le générateur est réinitialisé avant chaque run (`BasicRandomization.ResetSeed`), toutes les méthodes tournent à graine égale. Les nombres de la référence Python `cma` de Search-05b (ellipsoïde tourné : 3 850 évaluations jusqu'à $10^{-8}$) servent de cible d'ordre de grandeur, pas d'égalité stricte — les implémentations diffèrent sur les détails d'initialisation.

In [2]:
// References : les quatre DLL du domaine MetaGeneticSharp compilees en Release —
// GeneticSharp.Infrastructure.Framework (socle commun du moteur), GeneticSharp.Domain
// (algorithme genetique de base), MetaGeneticSharp.Infrastructure (utilitaires propres),
// MetaGeneticSharp.Domain (metaheuristiques geometriques). Le carnet branche le moteur
// local du depot, pas un paquet NuGet.
// Espaces de noms : GeneticSharp = moteur GA ; MetaGeneticSharp = metaheuristiques et
// benchmarks ; alias M = mathematiques NES, C = mathematiques CMA-ES.
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Release/net9.0/GeneticSharp.Infrastructure.Framework.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Release/net9.0/GeneticSharp.Domain.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Release/net9.0/MetaGeneticSharp.Infrastructure.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Release/net9.0/MetaGeneticSharp.Domain.dll"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Text;
using System.Net;
using System.Globalization;
using Microsoft.DotNet.Interactive.Formatting;
using GeneticSharp;
using MetaGeneticSharp;
using M = MetaGeneticSharp.NaturalEvolutionStrategyMath;
using C = MetaGeneticSharp.CovarianceMatrixAdaptationEvolutionStrategyMath;
Console.WriteLine($"MetaGeneticSharp {typeof(CmaEsDistribution).Assembly.GetName().Version}, GeneticSharp {typeof(GeneticSharp.IRandomization).Assembly.GetName().Version}");


MetaGeneticSharp 1.0.0.0, GeneticSharp 3.1.4.0


## Le chaînon manquant : rang-μ, rang un, pas cumulé

Search-05b a établi le lien au premier ordre : la mise à jour de rang-μ de CMA-ES **est** le gradient naturel de xNES sur la covariance — recombinaison pondérée des écarts des $\mu$ meilleurs à la moyenne. Le rang-μ est donc déjà couvert par la série. Ce que la classe `CovarianceMatrixAdaptationEvolutionStrategy` ajoute :

| Mécanisme | Ce qu'il fait | Pourquoi ça compte sur une vallée |
|---|---|---|
| **Rang un** ($p_c$, éq. 43 du tutoriel Hansen) | Cumule les déplacements successifs de la moyenne : $p_c \leftarrow (1-c_c)\,p_c + h_\sigma\sqrt{c_c(2-c_c)\,\mu_{\mathrm{eff}}}\,\Delta m$, puis replie $p_c p_c^\top$ dans $C$ au taux $c_1$ | La corrélation **entre générations** étire la covariance dans la direction où la moyenne progresse — une seule génération ne la voit pas |
| **CSA** ($p_\sigma$, éq. 44-46) | Compare la longueur du chemin dans l'espace *whitened* ($C^{-1/2}p_\sigma$) à son espérance $\mathbb{E}\|\mathcal{N}(0,I)\|$ : $\sigma$ croît si le chemin est long, décroît sinon | Le pas global suit la **trajectoire**, pas la seule population courante — aucun réglage de $\sigma$ à la main |
| **Restart-on-stall** | Après `StallGenerations` générations sans amélioration du meilleur jamais vu : covariance identité autour du meilleur point, $\sigma$ réinitialisé, compteur `Restarts` | Échappement des optima locaux multimodaux — plafond IPOP déclaré (population inchangée) |

Le témoin numérique de l'implémentation (test `Evaluations_ToReachOneMinusEight_OnARotatedEllipsoid_CMAEsVersusXNes`, graine 4242, cité dans la livraison de la phase 1) : CMA-ES atteint $10^{-8}$ en **2 269 évaluations** sur l'ellipsoïde tourné en d=5, xNES échoue dans le budget (meilleur $3{,}45\times 10^{-4}$). Le banc ci-dessous reproduit ce réglage exact, puis l'élargit à cinq bras et trois graines — dont l'ablation du seul rang un ($c_1$ neutralisé, tout le reste inchangé), qui doit pouvoir réfuter la lecture « le rang-un est le chaînon ».

In [3]:
// Le banc : transcription fidele du test de reference.
// tests/MetaGeneticSharp.Domain.Tests/Compound/CovarianceMatrixAdaptationEvolutionStrategyTests.cs
// (methodes Givens/Apply, ellipsoide tourne de Evaluations_ToReachOneMinusEight_...).
double[] Scales = { 1e4, 1e3, 1e2, 1e1, 1.0 };
int Dimension = Scales.Length;

// Matrice orthogonale produit de rotations de Givens, une par plan (i, i+1) -- verbatim du test.
double[,] Givens(IReadOnlyList<double> angles)
{
    int n = angles.Count + 1;
    var q = new double[n, n];
    for (int i = 0; i < n; i++) q[i, i] = 1.0;
    for (int plane = 0; plane < angles.Count; plane++)
    {
        var g = new double[n, n];
        for (int i = 0; i < n; i++) g[i, i] = 1.0;
        double c = Math.Cos(angles[plane]), s = Math.Sin(angles[plane]);
        g[plane, plane] = c; g[plane, plane + 1] = -s;
        g[plane + 1, plane] = s; g[plane + 1, plane + 1] = c;
        q = MatMul(q, g);
    }
    return q;
}
double[] Apply(double[,] m, double[] v)
{
    var r = new double[m.GetLength(0)];
    for (int i = 0; i < r.Length; i++)
    {
        double s = 0;
        for (int j = 0; j < v.Length; j++) s += m[i, j] * v[j];
        r[i] = s;
    }
    return r;
}
double[,] MatMul(double[,] a, double[,] b)
{
    var r = new double[a.GetLength(0), b.GetLength(1)];
    for (int i = 0; i < r.GetLength(0); i++)
        for (int j = 0; j < r.GetLength(1); j++)
        {
            double s = 0;
            for (int k = 0; k < a.GetLength(1); k++) s += a[i, k] * b[k, j];
            r[i, j] = s;
        }
    return r;
}
double RotatedEllipsoid(double[] x, double[,] q, double[] s)
{
    var r = Apply(q, x);
    double f = 0;
    for (int i = 0; i < s.Length; i++) f += s[i] * r[i] * r[i];
    return f;
}
double[,] Q5 = Givens(new[] { 0.6, -1.2, 0.35, 2.1 });
double angle01 = Math.Atan2(Q5[1, 0], Q5[0, 0]) * 180.0 / Math.PI;
Console.WriteLine($"d = {Dimension}, rapport d'echelles = {Scales[0]:0} (conditionnement de la forme quadratique)");
Console.WriteLine($"rotation du plan (x0,x1) = {angle01:F1} deg ; lambda papier(d=5) = {C.DefaultPopulationSize(Dimension)}");
Console.WriteLine("optimum global : x* = 0, f(x*) = 0 ; domaine initial [-10, 10]^5 (chromosome)");


d = 5, rapport d'echelles = 10000 (conditionnement de la forme quadratique)


rotation du plan (x0,x1) = 34,4 deg ; lambda papier(d=5) = 8


optimum global : x* = 0, f(x*) = 0 ; domaine initial [-10, 10]^5 (chromosome)


### Lecture du banc

Le paysage est sévère à dessein : vallée 5-D tournée de 34,4° dans le plan $(x_0, x_1)$, rapport d'échelles $10^4$ — le long axe de la forme quadratique est 10 000 fois plus raide que le court. La taille de population papier $\lambda(5) = 4 + \lfloor 3\ln 5\rfloor = 8$ est celle du conteneur CMA-ES ; le banc GA en utilise 30 pour donner à chacune des quatre méthodes la même largeur d'échantillonnage. L'optimum $x^\ast = 0$ est au centre du domaine $[-10,10]^5$ : aucune méthode ne peut le deviner par la position initiale, seule la géométrie apprise y mène.


In [4]:
// Le chromosome du banc : transcription verbatim de la classe privee du test
// (elle vit dans le fichier de test, pas dans la librairie).
/// <summary>A chromosome that randomises each gene in [min, max] on CreateNew, so the initial population is diverse.</summary>
sealed class RandomDoubleChromosome : ChromosomeBase
{
    private readonly double _min;
    private readonly double _max;

    public RandomDoubleChromosome(double min, double max, int length) : base(length)
    {
        _min = min;
        _max = max;
        var rnd = RandomizationProvider.Current;
        for (int i = 0; i < Length; i++)
            ReplaceGene(i, new Gene(_min + rnd.GetDouble() * (_max - _min)));
    }

    public override IChromosome CreateNew() => new RandomDoubleChromosome(_min, _max, Length);

    public override Gene GenerateGene(int geneIndex) =>
        new Gene(_min + RandomizationProvider.Current.GetDouble() * (_max - _min));

    public double[] GetDoubleValues() => GetGenes().Select(g => (double)g.Value).ToArray();
}

record GaOutcome(int Evals, double Best, int ReachedAt, List<(int Evals, double Best)> Trace);

// Le canevas GA commun : transcription du helper CountEvaluationsToTarget du test,
// generalisee a toute metaheuristique geometrique et instrumentee d'une trace par generation.
GaOutcome RunGa(GeometricMetaHeuristicBase strategy, int seed, int generations = 120, int popSize = 30, double target = 1e-8)
{
    strategy.SetGeometricConverter(new GeometricConverter<double>
    {
        GeneToDoubleConverter = (_, v) => v,
        DoubleToGeneConverter = (_, d) => d,
    });
    BasicRandomization.ResetSeed(seed);
    var previous = RandomizationProvider.Current;
    RandomizationProvider.Current = new BasicRandomization();
    try
    {
        var built = strategy.Build();
        var chromosome = new RandomDoubleChromosome(min: -10.0, max: 10.0, length: Dimension);
        int evaluations = 0, reachedAt = -1;
        double runningBest = double.PositiveInfinity;
        var trace = new List<(int, double)>();
        var fitness = new FuncFitness(c =>
        {
            evaluations++;
            double f = RotatedEllipsoid(((RandomDoubleChromosome)c).GetDoubleValues(), Q5, Scales);
            if (f < runningBest)
            {
                runningBest = f;
                if (f <= target && reachedAt < 0) reachedAt = evaluations;
            }
            return -f; // GeneticSharp maximise ; l'objectif minimise f
        });
        var population = new MetaPopulation(popSize, popSize, chromosome);
        var ga = new MetaGeneticAlgorithm(population, fitness, new EliteSelection(), new UniformCrossover(0.5f), new UniformMutation(true), built)
        {
            Termination = new GenerationNumberTermination(generations)
        };
        ga.GenerationRan += (_, _) => trace.Add((evaluations, runningBest));
        ga.Start();
        return new GaOutcome(evaluations, runningBest, reachedAt, trace);
    }
    finally
    {
        RandomizationProvider.Current = previous;
    }
}

// Passe de reference : CMA-ES, graine 4242 -- les nombres du test doivent revenir.
var sanity = RunGa(new CovarianceMatrixAdaptationEvolutionStrategy(), 4242);
Console.WriteLine($"CMA-ES graine 4242 : {sanity.Evals} evals, meilleur f = {sanity.Best:E3}, cible 1e-8 atteinte a {(sanity.ReachedAt > 0 ? sanity.ReachedAt.ToString() : "jamais")} evals");


CMA-ES graine 4242 : 3600 evals, meilleur f = 1,968E-016, cible 1e-8 atteinte a 2269 evals


## La démo discriminative : cinq bras, un seul canevas

Le protocole épouse la clause « démo discriminante » de l'issue d'onboarding : **même** population (30 individus), **même** budget (120 générations, 3 600 évaluations), **même** opérateurs du canevas (sélection élitiste, crossover uniforme, mutation — que CMA-ES remplace par ses échantillons de distribution), **même** graine par run. Seul varie le composé branché sur le moteur — et, pour le bras d'ablation, le seul commutateur qui neutralise le terme de rang un.

| Méthode | Composé | Ce qu'il modélise |
|---|---|---|
| **CMA-ES** | `CovarianceMatrixAdaptationEvolutionStrategy` | covariance pleine, rang un + rang-μ, CSA, restart-on-stall |
| **CMA-ES sans rang-un** | la même classe, `RankOneUpdate = false` | covariance pleine, rang-μ + CSA + restart **inchangés** : seul $c_1$ est neutralisé |
| **xNES** | `NaturalEvolutionStrategy` (`Full`) | covariance pleine par gradient naturel — le rang-μ seul |
| **SNES** | `NaturalEvolutionStrategy` (`Separable`) | covariance diagonale : aucun axe croisé ne peut être appris |
| **DE** | `DifferentialEvolution` | pas de distribution : différences de vecteurs courants |

Les deux premières lignes forment une **comparaison appariée** : même classe, même population, même budget, même graine, un seul coefficient de différence. xNES reste la **référence externe** — le rang-μ seul, sans chemin cumulé.

Trois graines {4242, 1717, 2718} ; la cible est $f \le 10^{-8}$, comptée à l'évaluation où le meilleur courant l'atteint pour la première fois.

In [5]:
// Les cinq bras compares, chacun instancie fraichement par graine :
// CMA-ES              — adaptation de la matrice de covariance complete (etalon de la famille) ;
// CMA-ES sans rang-un — la MEME classe, seul le terme de rang un neutralise
//                       (RankOneUpdate = false) : le rang-mu, le CSA, la politique de
//                       redemarrage, le budget et l'initialisation gardent leurs valeurs,
//                       donc l'ecart mesure isole ce terme et rien d'autre ;
// xNES                — covariance pleine par gradient naturel, soit le rang-mu seul ;
// SNES                — NES separable (covariance diagonale, une variance par dimension) ;
// DE                  — evolution differentielle, baseline classique hors famille NES/CMA.
var methods = new (string Name, Func<GeometricMetaHeuristicBase> Factory)[]
{
    ("CMA-ES", () => new CovarianceMatrixAdaptationEvolutionStrategy()),
    ("CMA-ES sans rang-un", () => new CovarianceMatrixAdaptationEvolutionStrategy { RankOneUpdate = false }),
    ("xNES",   () => new NaturalEvolutionStrategy { CovarianceModel = NesCovarianceModel.Full }),
    ("SNES",   () => new NaturalEvolutionStrategy { CovarianceModel = NesCovarianceModel.Separable }),
    ("DE",     () => new DifferentialEvolution()),
};
int[] Seeds = { 4242, 1717, 2718 };
var bench = new Dictionary<string, List<GaOutcome>>();
foreach (var (name, factory) in methods)
{
    var runs = new List<GaOutcome>();
    foreach (int seed in Seeds) runs.Add(RunGa(factory(), seed));
    bench[name] = runs;
    int reached = runs.Count(r => r.ReachedAt > 0);
    var medRun = runs.OrderBy(r => r.Best).ElementAt(runs.Count / 2);
    var medEvals = runs.Where(r => r.ReachedAt > 0).Select(r => (double)r.ReachedAt).DefaultIfEmpty(-1).OrderBy(e => e).ElementAt(Math.Max(0, (reached - 1) / 2));
    Console.WriteLine($"{name,-20} | {reached}/3 graines atteignent 1e-8 | mediane meilleur f = {medRun.Best,10:E3} | evals medianes jusqu'a la cible = {(medEvals > 0 ? ((int)medEvals).ToString() : "jamais")}");
}

// La comparaison appariee, graine par graine : le bras complet et son ablation
// partent du meme etat initial et du meme generateur, et ne different que par c_1.
// xNES reste la reference externe -- le rang-mu seul, sans terme de rang un.
var full = bench["CMA-ES"];
var ablated = bench["CMA-ES sans rang-un"];
var xnes = bench["xNES"];
string hSeed = "graine", hFull = "CMA-ES", hAbl = "sans rang-un", hDelta = "ecart (ordres)", hXnes = "xNES (ref.)", hReach = "atteint (complet / ablation)";
Console.WriteLine();
Console.WriteLine("Comparaison appariee par graine -- le meme CMA-ES, c_1 neutralise seul :");
Console.WriteLine($"{hSeed,-8} | {hFull,11} | {hAbl,11} | {hDelta,13} | {hXnes,11} | {hReach}");
bool identical = true;
for (int i = 0; i < Seeds.Length; i++)
{
    var rf = full[i]; var ra = ablated[i];
    identical &= ra.Best == rf.Best;
    double delta = Math.Log10(Math.Max(ra.Best, 1e-300)) - Math.Log10(Math.Max(rf.Best, 1e-300));
    string rfTxt = rf.ReachedAt > 0 ? rf.ReachedAt.ToString() : "jamais";
    string raTxt = ra.ReachedAt > 0 ? ra.ReachedAt.ToString() : "jamais";
    Console.WriteLine($"{Seeds[i],-8} | {rf.Best,11:E2} | {ra.Best,11:E2} | {delta,13:F2} | {xnes[i].Best,11:E2} | {rfTxt} / {raTxt}");
}

// Controle de non-vacuite : des bras qui rendraient exactement le meme meilleur
// final sur les trois graines signifieraient que le commutateur n'a rien change,
// et la comparaison ne dirait rien de l'effet du rang un.
Console.WriteLine();
Console.WriteLine(identical
    ? "CONTROLE : les deux bras rendent exactement le meme meilleur final sur les trois graines -- le commutateur n'a rien change, la comparaison serait vide."
    : "CONTROLE : les deux bras divergent sur au moins une graine -- la comparaison porte sur un effet reel du terme de rang un.");

CMA-ES               | 3/3 graines atteignent 1e-8 | mediane meilleur f = 1,968E-016 | evals medianes jusqu'a la cible = 2271


CMA-ES sans rang-un  | 3/3 graines atteignent 1e-8 | mediane meilleur f = 2,325E-014 | evals medianes jusqu'a la cible = 2611


xNES                 | 0/3 graines atteignent 1e-8 | mediane meilleur f = 3,447E-004 | evals medianes jusqu'a la cible = jamais


SNES                 | 0/3 graines atteignent 1e-8 | mediane meilleur f = 1,183E-001 | evals medianes jusqu'a la cible = jamais


DE                   | 0/3 graines atteignent 1e-8 | mediane meilleur f = 1,698E-002 | evals medianes jusqu'a la cible = jamais


Comparaison appariee par graine -- le meme CMA-ES, c_1 neutralise seul :


graine   |      CMA-ES | sans rang-un | ecart (ordres) | xNES (ref.) | atteint (complet / ablation)


4242     |   1,97E-016 |   1,80E-014 |          1,96 |   3,45E-004 | 2269 / 2522


1717     |   9,07E-016 |   2,04E-013 |          2,35 |   3,22E-004 | 2462 / 2792


2718     |   1,48E-016 |   2,33E-014 |          2,19 |   7,35E-004 | 2271 / 2611


CONTROLE : les deux bras divergent sur au moins une graine -- la comparaison porte sur un effet reel du terme de rang un.


### Lecture de la démo

D'abord la passe de référence : CMA-ES, graine 4242, atteint la cible $10^{-8}$ à **2 269 évaluations** avec un meilleur final à $1{,}97\times10^{-16}$ — les nombres exacts du test unitaire, au coup près (la trace par génération ajoute un comptage au fil de l'eau, sans toucher aux tirages). La transcription est fidèle.

Le banc élargi, lui, tranche net :

| Bras | Graines à $10^{-8}$ | Médiane meilleur $f$ | Évaluations médianes |
|---|---|---|---|
| **CMA-ES** | **3/3** | $1{,}97\times10^{-16}$ | **2 271** |
| CMA-ES sans rang-un | 3/3 | $2{,}33\times10^{-14}$ | 2 611 |
| xNES | 0/3 | $3{,}45\times10^{-4}$ | jamais |
| SNES | 0/3 | $1{,}18\times10^{-1}$ | jamais |
| DE | 0/3 | $1{,}70\times10^{-2}$ | jamais |

Quatre lectures, dans l'ordre du contraste :

- **L'écart de douze ordres de grandeur est réel — et l'ablation le réattribue.** CMA-ES ($10^{-16}$) domine xNES ($3{,}5\times10^{-4}$) de douze ordres ; pourtant neutraliser le seul $c_1$ ne détruit pas la performance : le bras sans rang-un atteint **3/3** la cible, à ~2 ordres de grandeur du bras complet et ~15 % d'évaluations en plus (2 611 vs 2 271 ; écarts par graine +1,96 / +2,35 / +2,19 ordres, jamais nuls — le contrôle de non-vacuité l'imprime). La lecture « le rang-un achète les douze ordres de grandeur » est donc **réfutée** : si le repli $p_c p_c^\top$ portait l'écart, le neutraliser aurait dû ramener CMA-ES vers le plateau de xNES ; il reste à $10^{-14}$ avec la cible atteinte. L'écart massif vit dans le reste du cadre — CSA, redémarrage sur stagnation, mise à jour additive de la covariance pleine — que cette ablation ne sépare pas entre eux.
- **Le rang-un n'en reste pas moins un vrai mécanisme, à l'échelle où il agit.** Sur runs appariés (même graine, même initialisation, un seul coefficient de différence), l'effet est systématique : trois graines sur trois dans le même sens, jamais nul. Il accélère la fin de course — une fois la vallée approximativement alignée par le rang-μ, le repli du chemin cumulé affine la covariance là où la recombinaison d'une seule génération reste bruitée. Deux ordres de grandeur et 15 % de budget : un raffinement, pas le chaînon.
- **Le modèle diagonal trompe plus que pas de modèle.** SNES ($1{,}2\times10^{-1}$) finit *derrière* DE ($1{,}7\times10^{-2}$) : une covariance diagonale apprend des échelles par axe de coordonnées alors que la vallée vit selon ses propres axes — le modèle contraint l'espace de recherche dans la direction fausse. DE, sans distribution, laisse ses différences de vecteurs suivre librement la pente locale.
- **Les trois graines concordent.** L'écart n'est pas un effet de graine : les médianes cachent des runs individuels tous cohérents (c'est ce que la figure suivante montre trajectoire par trajectoire).

In [6]:
// Figure : convergence log10(meilleur f) en fonction des evaluations, run median de chaque bras.
// Le bras complet et son ablation partagent la meme couleur de famille (vert / violet) :
// l'ecart entre les deux courbes est l'effet mesure du seul terme de rang un.
record PlotSvg(string Content);
Formatter.Register<PlotSvg>((p, w) => w.Write(p.Content), "text/html");
string Esc(string s) => WebUtility.HtmlEncode(s);
string F(double x) => x.ToString("F2", CultureInfo.InvariantCulture);
string DemoFigure()
{
    var sb = new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='900' height='430' viewBox='0 0 900 430'><rect width='900' height='430' fill='white'/>");
    int left = 70, w = 790, top = 45, h = 300;
    double maxEvals = bench.Values.SelectMany(r => r).Max(r => r.Evals);
    sb.Append($"<path d='M{left} {top} V{top + h} H{left + w}' fill='none' stroke='black'/>");
    sb.Append($"<text x='{left + w / 2 - 60}' y='415'>evaluations</text><text x='{left - 58}' y='{top + 15}'>log10 meilleur f</text>");
    for (int tick = -16; tick <= 8; tick += 4)
    {
        double y = top + h * (8.0 - tick) / 24.0;
        sb.Append($"<text x='{left - 32}' y='{F(y + 4)}'>{tick}</text><path d='M{left - 4} {F(y)} H{left}' stroke='black'/>");
    }
    foreach (int tick in new[] { 1000, 2000, 3000 })
    {
        double x = left + w * tick / maxEvals;
        sb.Append($"<text x='{F(x - 12)}' y='{F(top + h + 18)}'>{tick / 1000}k</text><path d='M{F(x)} {F(top + h)} v4' stroke='black'/>");
    }
    string[] colors = { "#21856b", "#7b3fa0", "#276fbf", "#c14953", "#8a6d3b" };
    int m = 0;
    foreach (var (name, _) in methods)
    {
        var run = bench[name].OrderBy(r => r.Best).ElementAt(bench[name].Count / 2); // run median
        var pts = string.Join(" ", run.Trace.Select(p => $"{F(left + w * p.Item1 / maxEvals)} {F(top + h * (8 - Math.Clamp(Math.Log10(Math.Max(p.Item2, 1e-17)), -16, 8)) / 24.0)}"));
        sb.Append($"<polyline fill='none' stroke='{colors[m]}' points='{pts}'/>");
        sb.Append($"<text x='{left + 10 + 140 * m}' y='{F(top + h + 42)}' fill='{colors[m]}'>{Esc(name)}</text>");
        m++;
    }
    double yT = top + h * (8 - Math.Log10(1e-8)) / 24.0;
    sb.Append($"<path d='M{left} {F(yT)} H{left + w}' stroke='#777' stroke-dasharray='4 3'/><text x='{left + w - 92}' y='{F(yT - 6)}'>cible 1e-8</text>");
    return sb.Append("</svg>").ToString();
}
display(new PlotSvg(DemoFigure()));

evaluations log10 meilleur f -16 -12 -8 -4 0 4 8 1k 2k 3k <polyline fill='none' stroke='#21856b' points='76.58 99.53 83.17 99.53 89.75 99.53 96.33 105.85 102.92 105.85 109.50 105.85 116.08 105.85 122.67 106.92 129.25 106.92 135.83 108.27 142.42 111.92 149.00 120.76 155.58 120.76 162.17 124.23 168.75 124.23 175.33 124.23 181.92 124.23 188.50 124.23 195.08 126.87 201.67 126.87 208.25 126.87 214.83 128.96 221.42 131.58 228.00 132.49 234.58 135.43 241.17 135.81 247.75 137.89 254.33 137.89 260.92 140.00 267.50 140.00 274.08 140.00 280.67 144.10 287.25 150.27 293.83 150.27 300.42 150.27 307.00 150.27 313.58 154.78 320.17 164.82 326.75 164.82 333.33 164.82 339.92 164.82 346.50 165.10 353.08 165.10 359.67 165.15 366.25 174.21 372.83 174.21 379.42 174.21 386.00 176.31 392.58 180.63 399.17 185.29 405.75 185.29 412.33 185.29 418.92 187.24 425.50 190.28 432.08 199.12 438.67 199.12 445.25 202.30 451.83 202.30 458.42 208.10 465.00 208.10 471.58 215.92 478.17 215.92 484.75 215.92 491.33 217.05 497.92 217.05 504.50 222.45 511.08 222.45 517.67 228.93 524.25 228.93 530.83 230.49 537.42 237.46 544.00 237.46 550.58 237.46 557.17 239.04 563.75 244.43 570.33 249.93 576.92 252.61 583.50 252.61 590.08 253.13 596.67 253.13 603.25 258.89 609.83 260.56 616.42 261.57 623.00 264.16 629.58 269.76 636.17 269.76 642.75 269.76 649.33 277.04 655.92 277.04 662.50 277.04 669.08 283.96 675.67 283.96 682.25 285.70 688.83 289.56 695.42 289.56 702.00 291.25 708.58 292.95 715.17 297.02 721.75 302.28 728.33 302.28 734.92 303.14 741.50 303.23 748.08 309.98 754.67 311.76 761.25 311.76 767.83 312.26 774.42 312.26 781.00 314.65 787.58 316.00 794.17 320.36 800.75 324.06 807.33 329.11 813.92 329.11 820.50 332.25 827.08 332.25 833.67 332.25 840.25 335.05 846.83 337.04 853.42 338.87 860.00 341.32'/> CMA-ES <polyline fill='none' stroke='#7b3fa0' points='76.58 93.03 83.17 93.03 89.75 97.32 96.33 104.45 102.92 104.45 109.50 105.48 116.08 108.71 122.67 114.49 129.25 114.49 135.83 114.49 142.42 114.49 149.00 114.49 155.58 114.49 162.17 117.54 168.75 117.54 175.33 120.70 181.92 120.70 188.50 120.70 195.08 123.53 201.67 123.53 208.25 134.55 214.83 134.55 221.42 134.55 228.00 134.55 234.58 134.55 241.17 134.55 247.75 134.55 254.33 134.55 260.92 135.38 267.50 137.85 274.08 137.85 280.67 140.57 287.25 140.57 293.83 140.57 300.42 142.75 307.00 143.86 313.58 143.86 320.17 143.86 326.75 147.97 333.33 150.47 339.92 150.47 346.50 150.47 353.08 152.36 359.67 160.90 366.25 160.90 372.83 160.90 379.42 160.90 386.00 162.48 392.58 163.84 399.17 167.74 405.75 167.74 412.33 167.74 418.92 169.28 425.50 171.12 432.08 176.12 438.67 176.38 445.25 180.34 451.83 180.34 458.42 180.34 465.00 181.19 471.58 184.01 478.17 185.68 484.75 187.22 491.33 190.00 497.92 193.30 504.50 193.34 511.08 200.92 517.67 201.12 524.25 205.49 530.83 208.74 537.42 208.74 544.00 209.80 550.58 213.65 557.17 215.56 563.75 215.81 570.33 222.94 576.92 222.94 583.50 224.18 590.08 224.18 596.67 226.12 603.25 226.47 609.83 232.15 616.42 235.35 623.00 235.83 629.58 238.64 636.17 239.33 642.75 241.44 649.33 245.98 655.92 248.60 662.50 249.08 669.08 249.08 675.67 262.62 682.25 262.62 688.83 262.62 695.42 263.93 702.00 267.39 708.58 267.39 715.17 267.90 721.75 269.60 728.33 275.76 734.92 277.40 741.50 277.40 748.08 280.45 754.67 281.74 761.25 283.42 767.83 287.65 774.42 298.87 781.00 298.87 787.58 298.87 794.17 298.87 800.75 298.87 807.33 298.87 813.92 303.82 820.50 305.31 827.08 306.02 833.67 310.75 840.25 310.75 846.83 313.69 853.42 315.42 860.00 315.42'/> CMA-ES sans rang-un <polyline fill='none' stroke='#276fbf' points='76.58 99.53 83.17 99.53 89.75 99.53 96.33 99.53 102.92 100.24 109.50 100.24 116.08 100.24 122.67 103.36 129.25 103.36 135.83 103.36 142.42 103.36 149.00 104.82 155.58 104.82 162.17 107.15 168.75 107.15 175.33 107.15 181.92 107.25 188.50 107.25 195.08 109.30 201.67 109.30 208.25 109.30 214.83 112.14 221.42 112.14 228.00 117.09 234.58 117.09 241.17 117.09 247.75 117.09 254.33 117.09 260.92 117.09 267.50 117.09 274.08 117

### Lecture de la figure

Chaque courbe est le run **médian** de son bras (classement par meilleur final). Deux courbes franchissent la ligne pointillée de la cible $10^{-8}$ : CMA-ES vers 2 300 évaluations, et son ablation du rang-un un peu plus tard, vers 2 600 — puis la courbe complète continue de descendre jusqu'à $10^{-16}$ quand l'ablation s'arrête deux ordres de grandeur plus haut. Les trois autres courbes s'aplatissent : xNES stabilise autour de $10^{-3{,}5}$, DE vers $10^{-2}$, SNES vers $10^{-1}$ — leurs plateaux sont le prix du mécanisme manquant, pas un manque de budget (les 3 600 évaluations du budget sont toutes consommées).

## La covariance apprend l'orientation (d = 2)

Sur l'ellipsoïde tourné 5-D du banc, la covariance vit dans un espace qu'on ne peut pas dessiner. Le même mécanisme se lit en 2-D : vallée tournée par `Givens(0.9)` dans le plan, rapports d'échelles $\{25, 1\}$. Cette fois la distribution est pilotée **directement** (`CmaEsDistribution.Create` / `Sample` / `UpdateWithFitness`) : une boucle par génération, $\lambda = $ `DefaultPopulationSize(2)` échantillons, la population entière classée sert à la mise à jour — exactement ce que le conteneur exécute à l'intérieur du GA.

Deux quantités relevées toutes les 5 générations :

- le **rapport d'axes** $\sqrt{\lambda_{\max}/\lambda_{\min}}$ de la covariance — la cible géométrique est le rapport des demi-axes de la vallée, $\sqrt{25/1} = 5$ ;
- l'**angle** de l'axe principal — la cible est l'orientation du vecteur propre mineur de $Q^\top \mathrm{diag}(s)\,Q$, calculée en fermé.

C'est la réponse algébrique à l'exercice 3 de MGS-07d : là-bas, le rapport vertical/horizontal mesuré sur la heatmap *révélait* une vallée orientée ; ici, la covariance de recherche *apprend* cette orientation — sans qu'on la lui donne.


In [7]:
// Distribution CMA-ES pilotee directement, d=2 : la covariance apprend l'orientation.
double[] S2 = { 25.0, 1.0 };
double[,] Q2 = Givens(new[] { 0.9 });
double F2(double[] x) => RotatedEllipsoid(x, Q2, S2);

// Orientation theorique de la vallee : vecteur propre mineur de H = Q^T diag(s) Q (ferme en 2-D).
double[,] Diag(double[] s) { var d = new double[s.Length, s.Length]; for (int i = 0; i < s.Length; i++) d[i, i] = s[i]; return d; }
double[,] Transpose(double[,] a) { var t = new double[a.GetLength(1), a.GetLength(0)]; for (int i = 0; i < a.GetLength(0); i++) for (int j = 0; j < a.GetLength(1); j++) t[j, i] = a[i, j]; return t; }
var H = MatMul(MatMul(Transpose(Q2), Diag(S2)), Q2);
double trH = H[0, 0] + H[1, 1], detH = H[0, 0] * H[1, 1] - H[0, 1] * H[1, 0];
double lamMin = (trH - Math.Sqrt(trH * trH - 4 * detH)) / 2;
double Fold90(double deg) { deg = Math.Abs(deg) % 180.0; if (deg > 90.0) deg = 180.0 - deg; return deg; }
double valleyAngle = Fold90(Math.Atan2(lamMin - H[0, 0], H[0, 1]) * 180.0 / Math.PI);

int lambda2 = C.DefaultPopulationSize(2);
BasicRandomization.ResetSeed(4242);
var rng2 = new BasicRandomization();
var dist = CmaEsDistribution.Create(new[] { 4.0, -3.0 }, 1.5, M.Identity(2));
var eigLog = new List<(int Gen, double Ratio, double AngleDeg, double Sigma)>();
var snaps = new List<(int Gen, double[] Mean, double[] Ev, double[,] Vec, double Sigma)>();
for (int gen = 0; gen < 160; gen++)
{
    var samples = new List<(double[] X, double F)>();
    for (int k = 0; k < lambda2; k++) { var x = dist.Sample(rng2); samples.Add((x, F2(x))); }
    samples = samples.OrderBy(p => p.F).ToList(); // meilleure en tete
    dist = dist.UpdateWithFitness(samples.Select(p => p.X).ToList(), samples.Select(p => -p.F).ToList());
    if (gen % 5 == 0 || gen == 159)
    {
        var ev = dist.Eigenvalues;
        int imax = ev[0] >= ev[1] ? 0 : 1, imin = 1 - imax;
        eigLog.Add((gen, Math.Sqrt(ev[imax] / ev[imin]), Fold90(Math.Atan2(dist.Eigenvectors[1, imax], dist.Eigenvectors[0, imax]) * 180.0 / Math.PI), dist.Sigma));
    }
    if (gen is 0 or 25 or 50 or 100 or 159)
        snaps.Add((gen, (double[])dist.Mean.Clone(), (double[])dist.Eigenvalues.Clone(), (double[,])dist.Eigenvectors.Clone(), dist.Sigma));
}
var lastEig = eigLog[^1];
Console.WriteLine($"lambda(2) = {lambda2} ; orientation theorique de la vallee = {valleyAngle:F1} deg ; rapport d'axes cible = {Math.Sqrt(S2[0] / S2[1]):F1}");
Console.WriteLine($"generation   0 : rapport d'axes = {eigLog[0].Ratio:F2}, angle = {eigLog[0].AngleDeg:F1} deg, sigma = {eigLog[0].Sigma:F3}");
Console.WriteLine($"generation 159 : rapport d'axes = {lastEig.Ratio:F2}, angle = {lastEig.AngleDeg:F1} deg, sigma = {lastEig.Sigma:F3}, meilleur f = {-dist.BestFitness:E3}");
var at35 = eigLog.First(e => e.Gen == 35);
Console.WriteLine($"generation  35 : rapport d'axes = {at35.Ratio:F2}, angle = {at35.AngleDeg:F1} deg");
var maxEig = eigLog.OrderByDescending(e => e.Ratio).First();
Console.WriteLine($"rapport d'axes maximal = {maxEig.Ratio:F2} a la generation {maxEig.Gen}, puis retour a {lastEig.Ratio:F2} a la generation 159");


lambda(2) = 6 ; orientation theorique de la vallee = 38,4 deg ; rapport d'axes cible = 5,0


generation   0 : rapport d'axes = 1,18, angle = 49,2 deg, sigma = 2,302


generation 159 : rapport d'axes = 3,75, angle = 39,0 deg, sigma = 0,000, meilleur f = 9,888E-028


generation  35 : rapport d'axes = 5,72, angle = 35,5 deg


rapport d'axes maximal = 8,79 a la generation 85, puis retour a 3,75 a la generation 159


### Lecture de la covariance apprise

Les deux quantités convergent vers la géométrie de la vallée, chacune à son rythme :

- **L'angle se verrouille en premier.** Bruyant les dix premières générations (l'estimation d'un axe à partir de 6 points est fragile), il se cale vers la génération 25 dans la bande 37-40°, et la cible théorique 38,4° y reste jusqu'au bout. C'est la réponse algébrique à l'exercice 3 de MGS-07d : l'anisotropie vertical/horizontal que la heatmap *révélait*, la covariance *l'apprend* — même rotation, même angle, mais cette fois du côté recherche.
- **Le rapport d'axes croise sa cible vers la génération 35** (5,72 mesuré contre $\sqrt{25/1} = 5$ attendu), dépasse jusqu'à 8,79 vers la génération 85, puis se fige : quand $\sigma$ s'effondre (convergence, meilleur $f$ à $10^{-28}$), les écarts échantillonnés — proportionnels à $\sigma$ — ne portent plus de signal, et le rapport mesuré dérive par bruit numérique d'affichage. L'apprentissage ne s'arrête pas parce qu'il a fini : il s'arrête parce que la recherche a convergé.

Autrement dit : les deux mesures racontent le même événement depuis deux côtés — la distribution devient la vallée, puis se réduit à un point.


In [8]:
// Figure : isovaleurs de la vallee + ellipses de recherche (2 sigma) a cinq generations ;
// panneau droit : convergence du rapport d'axes et de l'angle vers les valeurs geometriques.
string CovFigure()
{
    var sb = new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='1040' height='470' viewBox='0 0 1040 470'><rect width='1040' height='470' fill='white'/>");
    // Panneau gauche : le plan (x0, x1), vue [-7, 9] x [-7, 7].
    int L = 60, W = 430, T = 40, Hh = 380;
    double xMin = -7, xMax = 9, yMin = -7, yMax = 7;
    double PX(double x) => L + W * (x - xMin) / (xMax - xMin);
    double PY(double y) => T + Hh * (yMax - y) / (yMax - yMin);
    sb.Append($"<text x='{L}' y='25'>vallée tournée (isovaleurs) et ellipses de recherche 2σ</text>");
    sb.Append($"<path d='M{L} {T} V{T + Hh} H{L + W}' fill='none' stroke='black'/>");
    // Isovaleurs de la forme quadratique H (centree en 0) : decomposition propre analytique 2x2.
    double a = H[0, 0], b = H[0, 1], c2 = H[1, 1];
    double muMaj = (a + c2 + Math.Sqrt((a - c2) * (a - c2) + 4 * b * b)) / 2;
    double muMin = (a + c2 - Math.Sqrt((a - c2) * (a - c2) + 4 * b * b)) / 2;
    (double X, double Y) Norm2(double x, double y) { double n = Math.Sqrt(x * x + y * y); return n > 0 ? (x / n, y / n) : (1, 0); }
    var vMaj = Norm2(b, muMaj - a);
    var vMin = Norm2(b, muMin - a);
    foreach (double lev in new[] { 1.0, 10.0, 50.0, 200.0 })
    {
        var pts = new List<string>();
        for (int deg = 0; deg <= 360; deg += 6)
        {
            double th = deg * Math.PI / 180.0;
            double px = Math.Sqrt(lev / muMaj) * Math.Cos(th) * vMaj.X + Math.Sqrt(lev / muMin) * Math.Sin(th) * vMin.X;
            double py = Math.Sqrt(lev / muMaj) * Math.Cos(th) * vMaj.Y + Math.Sqrt(lev / muMin) * Math.Sin(th) * vMin.Y;
            pts.Add($"{F(PX(px))} {F(PY(py))}");
        }
        sb.Append($"<polygon fill='none' stroke='#c8d4e0' points='{string.Join(" ", pts)}'/>");
    }
    // Ellipses de recherche 2 sigma aux generations snapshot.
    string[] ecol = { "#b7e1cd", "#7cc7a3", "#4aa87f", "#2a8a63", "#0e6b4c" };
    int ci = 0;
    foreach (var s in snaps)
    {
        var pts = new List<string>();
        for (int deg = 0; deg <= 360; deg += 6)
        {
            double th = deg * Math.PI / 180.0;
            double px = s.Mean[0] + 2 * s.Sigma * (Math.Sqrt(s.Ev[0]) * Math.Cos(th) * s.Vec[0, 0] + Math.Sqrt(s.Ev[1]) * Math.Sin(th) * s.Vec[0, 1]);
            double py = s.Mean[1] + 2 * s.Sigma * (Math.Sqrt(s.Ev[0]) * Math.Cos(th) * s.Vec[1, 0] + Math.Sqrt(s.Ev[1]) * Math.Sin(th) * s.Vec[1, 1]);
            pts.Add($"{F(PX(px))} {F(PY(py))}");
        }
        sb.Append($"<polygon fill='none' stroke='{ecol[ci]}' stroke-width='2' points='{string.Join(" ", pts)}'/>");
        sb.Append($"<circle cx='{F(PX(s.Mean[0]))}' cy='{F(PY(s.Mean[1]))}' r='2.5' fill='{ecol[ci]}'/>");
        sb.Append($"<text x='{L + 8}' y='{T + 16 + 14 * ci}' fill='{ecol[ci]}'>gen {s.Gen}</text>");
        ci++;
    }
    // Panneau droit haut : rapport d'axes (cible 5).
    int L2 = 545, W2 = 440, T2 = 40, H2 = 170;
    sb.Append($"<text x='{L2}' y='25'>rapport d'axes sqrt(λmax/λmin) — cible 5</text>");
    sb.Append($"<path d='M{L2} {T2} V{T2 + H2} H{L2 + W2}' fill='none' stroke='black'/>");
    double RP(double v) => T2 + H2 * (9.0 - Math.Clamp(v, 0, 9)) / 9.0;
    foreach (int tick in new[] { 0, 3, 6, 9 }) sb.Append($"<text x='{L2 - 22}' y='{F(RP(tick) + 4)}'>{tick}</text><path d='M{L2 - 4} {F(RP(tick))} H{L2}' stroke='black'/>");
    double yT5 = RP(5);
    sb.Append($"<path d='M{L2} {F(yT5)} H{L2 + W2}' stroke='#777' stroke-dasharray='4 3'/><text x='{L2 + W2 - 60}' y='{F(yT5 - 5)}'>cible 5</text>");
    var rpts = string.Join(" ", eigLog.Select(e => $"{F(L2 + W2 * e.Gen / 159.0)} {F(RP(e.Ratio))}"));
    sb.Append($"<polyline fill='none' stroke='#21856b' points='{rpts}'/>");
    // Panneau droit bas : angle de l'axe principal (cible = orientation de la vallee).
    int T3 = 265, H3 = 155;
    sb.Append($"<text x='{L2}' y='250'>angle de l'axe principal (deg) — cible {valleyAngle:F1}</text>");
    sb.Append($"<path d='M{L2} {T3} V{T3 + H3} H{L2 + W2}' fill='none' stroke='black'/>");
    double AP(double v) => T3 + H3 * (90.0 - Math.Clamp(v, 0, 90)) / 90.0;
    foreach (int tick in new[] { 0, 45, 90 }) sb.Append($"<text x='{L2 - 22}' y='{F(AP(tick) + 4)}'>{tick}</text><path d='M{L2 - 4} {F(AP(tick))} H{L2}' stroke='black'/>");
    double yTv = AP(valleyAngle);
    sb.Append($"<path d='M{L2} {F(yTv)} H{L2 + W2}' stroke='#777' stroke-dasharray='4 3'/>");
    var apts = string.Join(" ", eigLog.Select(e => $"{F(L2 + W2 * e.Gen / 159.0)} {F(AP(e.AngleDeg))}"));
    sb.Append($"<polyline fill='none' stroke='#c14953' points='{apts}'/>");
    sb.Append($"<text x='{L2 + 10}' y='{F(T3 + H3 + 30)}' fill='#21856b'>rapport d'axes</text><text x='{L2 + 140}' y='{F(T3 + H3 + 30)}' fill='#c14953'>angle</text>");
    return sb.Append("</svg>").ToString();
}
display(new PlotSvg(CovFigure()));


vallée tournée (isovaleurs) et ellipses de recherche 2σ <polygon fill='none' stroke='#c8d4e0' points='244.78 225.75 242.60 227.53 240.48 229.35 238.44 231.17 236.51 232.98 234.71 234.75 233.05 236.48 231.56 238.13 230.24 239.69 229.13 241.15 228.22 242.49 227.53 243.68 227.07 244.73 226.84 245.62 226.84 246.34 227.07 246.87 227.54 247.22 228.23 247.39 229.14 247.36 230.25 247.14 231.56 246.74 233.06 246.15 234.72 245.38 236.52 244.45 238.45 243.36 240.49 242.12 242.61 240.75 244.80 239.26 247.02 237.67 249.25 235.99 251.47 234.25 253.65 232.47 255.77 230.65 257.81 228.83 259.74 227.02 261.54 225.25 263.20 223.52 264.69 221.87 266.01 220.31 267.12 218.85 268.03 217.51 268.72 216.32 269.18 215.27 269.41 214.38 269.41 213.66 269.18 213.13 268.71 212.78 268.02 212.61 267.11 212.64 266.00 212.86 264.69 213.26 263.19 213.85 261.53 214.62 259.73 215.55 257.80 216.64 255.76 217.88 253.64 219.25 251.45 220.74 249.23 222.33 247.00 224.01 244.78 225.75'/><polygon fill='none' stroke='#c8d4e0' points='237.56 216.55 230.66 222.20 223.95 227.94 217.50 233.70 211.40 239.42 205.69 245.03 200.45 250.48 195.73 255.71 191.58 260.65 188.06 265.26 185.19 269.48 183.01 273.27 181.55 276.59 180.81 279.39 180.81 281.66 181.55 283.35 183.02 284.47 185.20 284.98 188.08 284.90 191.61 284.21 195.75 282.93 200.48 281.07 205.72 278.65 211.43 275.69 217.54 272.24 223.99 268.32 230.70 263.99 237.60 259.28 244.62 254.25 251.67 248.95 258.69 243.45 265.59 237.80 272.30 232.06 278.75 226.30 284.85 220.58 290.56 214.97 295.80 209.52 300.52 204.29 304.67 199.35 308.19 194.74 311.06 190.52 313.24 186.73 314.70 183.41 315.44 180.61 315.44 178.34 314.70 176.65 313.23 175.53 311.05 175.02 308.17 175.10 304.64 175.79 300.50 177.07 295.77 178.93 290.53 181.35 284.82 184.31 278.71 187.76 272.26 191.68 265.55 196.01 258.65 200.72 251.63 205.75 244.58 211.05 237.56 216.55'/><polygon fill='none' stroke='#c8d4e0' points='224.50 199.93 209.07 212.57 194.07 225.39 179.66 238.27 166.00 251.06 153.23 263.61 141.51 275.80 130.96 287.49 121.69 298.54 113.81 308.85 107.40 318.29 102.53 326.76 99.25 334.17 97.61 340.45 97.61 345.51 99.27 349.30 102.55 351.79 107.43 352.95 113.85 352.76 121.74 351.22 131.02 348.36 141.58 344.19 153.31 338.78 166.08 332.18 179.74 324.45 194.16 315.69 209.16 305.99 224.59 295.46 240.28 284.22 256.06 272.37 271.75 260.07 287.18 247.43 302.18 234.61 316.59 221.73 330.25 208.94 343.02 196.39 354.74 184.20 365.29 172.51 374.56 161.46 382.44 151.15 388.85 141.71 393.72 133.24 397.00 125.83 398.64 119.55 398.64 114.49 396.98 110.70 393.70 108.21 388.82 107.05 382.40 107.24 374.51 108.78 365.23 111.64 354.67 115.81 342.94 121.22 330.17 127.82 316.51 135.55 302.09 144.31 287.09 154.01 271.66 164.54 255.97 175.78 240.19 187.63 224.50 199.93'/><polygon fill='none' stroke='#c8d4e0' points='200.87 169.86 170.01 195.13 140.01 220.79 111.19 246.54 83.87 272.11 58.34 297.22 34.90 321.60 13.80 344.97 -4.74 367.08 -20.51 387.69 -33.33 406.57 -43.07 423.52 -49.62 438.35 -52.91 450.89 -52.90 461.02 -49.59 468.61 -43.02 473.59 -33.26 475.90 -20.42 475.51 -4.64 472.44 13.92 466.71 35.04 458.39 58.49 447.56 84.03 434.35 111.36 418.90 140.19 401.39 170.20 381.99 201.06 360.93 232.44 338.43 264.00 314.75 295.38 290.14 326.24 264.87 356.24 239.21 385.06 213.46 412.38 187.89 437.91 162.78 461.35 138.40 482.45 115.03 500.99 92.92 516.76 72.31 529.58 53.43 539.32 36.48 545.87 21.65 549.16 9.11 549.15 -1.02 545.84 -8.61 539.27 -13.59 529.51 -15.90 516.67 -15.51 500.89 -12.44 482.33 -6.71 461.21 1.61 437.76 12.44 412.22 25.65 384.89 41.10 356.06 58.61 326.05 78.01 295.19 99.07 263.81 121.57 232.25 145.25 200.87 169.86'/><polygon fill='none' stroke='#b7e1cd' stroke-width='2' points='378.77 155.39 368.62 144.54 357.60 134.68 345.81 125.90 333.39 118.32 320.48 112.01 307.21 107.03 293.74 103.46 280.20 101.32 266.75 100.63 253.53 101.42 240.70 103.66 228.38 107.33 216.72 112.39 205.85 118.79 195.87 126.46 186.91 135.31 179.05 145.25 172.39 156.16 167.00 167.93 162.94 180.43 160.25 193.

### Lecture de la figure

À gauche, le plan de la vallée : isovaleurs grises de la forme quadratique, et les ellipses de recherche $2\sigma$ aux cinq générations relevées. L'ellipse de départ est large et encore presque ronde ; dès la génération 25 elle s'allonge le long de la vallée ; à partir de la génération 50 elle n'est plus qu'un point autour de l'optimum — $\sigma$ s'est effondré, la recherche a convergé. À droite, la version quantitative : le rapport d'axes (vert) croise la ligne pointillée de la cible 5 vers la génération 30-35 ; l'angle (rouge) se cale sur la cible 38,4° après une dizaine de générations d'agitation initiale et n'en bouge plus.


## CSA : le pas suit la trajectoire ; la stagnation déclenche le redémarrage

Deux mesures, deux paysages :

1. **Trajectoire de $\sigma$** (CSA) — relevée dans le run 2-D précédent (`eigLog`) : $\sigma$ en fonction de la génération. La question : le pas global traduit-il la trajectoire de la moyenne — maintenu grand tant que la progression est régulière, resserré quand la vallée se resserre ?
2. **Restart-on-stall** — sur Rastrigin 5-D (multimodal, optimum global $0$ en $x^\ast = 0$), départ à $(4,4,4,4,4)$, $\sigma_0 = 1$ : la boucle appelle `Restart` dès que `GenerationsWithoutImprovement` atteint 40 — la mécanique exacte que la propriété `StallGenerations` du compose déclenche côté GA. On trace $\sigma$ et les instants de redémarrage.

**Plafond IPOP, déclaré par la classe** (remarques de `CovarianceMatrixAdaptationEvolutionStrategy`, verbatim) : « *restart with doubled population is not reachable from inside a compound here: the population size belongs to the `MetaGeneticAlgorithm` and cannot change mid-run. A caller that wants IPOP proper can read `DefaultPopulationSize` and build the next run with a multiplied population. This ceiling is declared rather than papered over: the tests measure restart-on-stall, not IPOP.* » L'orchestrateur de redémarrages à populations croissantes est livré à part dans le sous-module (PR jsboige/MetaGeneticSharp#75, tranche partielle : intégration automatique et budget global restent ouverts) — l'exercice 1 construit la boucle côté appelant, comme la citation le prescrit.


In [9]:
// (1) Trajectoire sigma du run 2-D (relevee dans eigLog).
Console.WriteLine("sigma par generation (vallee 2-D) :");
foreach (var e in eigLog.Where((_, i) => i % 2 == 0).Take(8))
    Console.WriteLine($"  gen {e.Gen,3} : sigma = {e.Sigma:F4}");
Console.WriteLine($"  ... gen {eigLog[^1].Gen,3} : sigma = {eigLog[^1].Sigma:F4}");

// (2) Restart-on-stall sur Rastrigin 5-D, distribution pilotee ( meme graine ).
double Rastrigin(double[] x)
{
    double f = 10.0 * x.Length;
    foreach (double xi in x) f += xi * xi - 10.0 * Math.Cos(2.0 * Math.PI * xi);
    return f;
}
int dR = 5, lambdaR = C.DefaultPopulationSize(dR), stallLimit = 40, maxGenR = 400;
BasicRandomization.ResetSeed(4242);
var rngR = new BasicRandomization();
var distR = CmaEsDistribution.Create(Enumerable.Repeat(4.0, dR).ToArray(), 1.0, M.Identity(dR));
var sigmaTrace = new List<(int Gen, double Sigma, int Restarts, double BestF)>();
for (int gen = 0; gen < maxGenR; gen++)
{
    var samples = new List<(double[] X, double F)>();
    for (int k = 0; k < lambdaR; k++) { var x = distR.Sample(rngR); samples.Add((x, Rastrigin(x))); }
    samples = samples.OrderBy(p => p.F).ToList();
    distR = distR.UpdateWithFitness(samples.Select(p => p.X).ToList(), samples.Select(p => -p.F).ToList());
    if (distR.GenerationsWithoutImprovement >= stallLimit)
        distR = distR.Restart(distR.BestPoint, 1.0);
    sigmaTrace.Add((gen, distR.Sigma, distR.Restarts, -distR.BestFitness));
}
Console.WriteLine($"\nRastrigin d=5, lambda = {lambdaR}, stall = {stallLimit} : {distR.Restarts} redemarrages en {maxGenR} generations");
Console.WriteLine($"meilleur f final = {-distR.BestFitness:F3} (optimum global 0) ; meilleur point |x|_inf = {distR.BestPoint.Max(v => Math.Abs(v)):F3}");


sigma par generation (vallee 2-D) :


  gen   0 : sigma = 2,3018


  gen  10 : sigma = 1,1668


  gen  20 : sigma = 0,5632


  gen  30 : sigma = 0,5558


  gen  40 : sigma = 0,1906


... (2 outputs left out)

  gen  70 : sigma = 0,0012


  ... gen 159 : sigma = 0,0000



Rastrigin d=5, lambda = 8, stall = 40 : 3 redemarrages en 400 generations


meilleur f final = 5,970 (optimum global 0) ; meilleur point |x|_inf = 1,990


### Lecture des trajectoires de pas

Deux paysages, deux comportements du même CSA :

- **Vallée unimodale 2-D :** $\sigma$ décroît de 2,30 à presque zéro en une cinquantaine de générations — dès que la trajectoire de la moyenne devient régulière dans la vallée, le chemin whitened raccourcit, et le pas se resserre. Pas de palier, pas d'oscillation : la décroissance suit la convergence (meilleur $f$ final à $9{,}9\times10^{-28}$).
- **Rastrigin 5-D :** le même CSA mène au même effondrement — mais ici l'effondrement est le problème (la distribution s'enferme dans un bassin local). **3 redémarrages** en 400 générations (générations ~121, ~161, ~201, soit des intervalles de 40 = `stallLimit`) : chacun ré-initialise la covariance autour du meilleur point et ré-injecte $\sigma = 1$ — c'est le saut visible sur la figure. Le meilleur passe de $f \approx 80$ au départ $(4,\ldots,4)$ à **5,97** ($|x|_\infty = 1{,}99$ : le meilleur point vit dans un bassin voisin de l'optimum global, pas encore dedans). Le redémarrage exploré ici est *restart-on-stall* à population constante — le plafond IPOP déclaré plus haut reste à lever côté appelant (exercice 1).


In [10]:
// Figure : sigma(generation) sur Rastrigin avec les instants de redemarrage ;
// a droite, sigma sur la vallee 2-D (CSA seul, sans restart).
string CsaFigure()
{
    var sb = new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='1040' height='420' viewBox='0 0 1040 420'><rect width='1040' height='420' fill='white'/>");
    // Gauche : Rastrigin, sigma en log10.
    int L = 65, W = 430, T = 45, Hh = 300;
    sb.Append($"<text x='{L}' y='25'>Rastrigin d=5 : log10 sigma, redémarrages marqués</text>");
    sb.Append($"<path d='M{L} {T} V{T + Hh} H{L + W}' fill='none' stroke='black'/>");
    double LP(double v) => T + Hh * (1.0 - (Math.Clamp(Math.Log10(Math.Max(v, 1e-4)), -3, 1) + 3) / 4.0);
    for (int tick = -3; tick <= 1; tick++)
        sb.Append($"<text x='{L - 30}' y='{F(LP(Math.Pow(10, tick)) + 4)}'>{tick}</text><path d='M{L - 4} {F(LP(Math.Pow(10, tick)))} H{L}' stroke='black'/>");
    foreach (int tick in new[] { 100, 200, 300 })
    {
        double x = L + W * tick / 400.0;
        sb.Append($"<text x='{F(x - 12)}' y='{F(T + Hh + 18)}'>{tick}</text><path d='M{F(x)} {F(T + Hh)} v4' stroke='black'/>");
    }
    int prevRestarts = 0;
    foreach (var t in sigmaTrace)
    {
        if (t.Restarts > prevRestarts)
        {
            double x = L + W * t.Gen / 400.0;
            sb.Append($"<path d='M{F(x)} {T} V{T + Hh}' stroke='#c14953' stroke-dasharray='3 3'/>");
            prevRestarts = t.Restarts;
        }
    }
    var spts = string.Join(" ", sigmaTrace.Where((_, i) => i % 2 == 0).Select(t => $"{F(L + W * t.Gen / 400.0)} {F(LP(t.Sigma))}"));
    sb.Append($"<polyline fill='none' stroke='#276fbf' points='{spts}'/>");
    sb.Append($"<text x='{L + 10}' y='{T + Hh + 40}' fill='#276fbf'>sigma</text><text x='{L + 80}' y='{T + Hh + 40}' fill='#c14953'>| restart-on-stall</text>");
    // Droite : vallee 2-D, sigma lineaire.
    int L2 = 560, W2 = 430, T2 = 45, H2 = 300;
    sb.Append($"<text x='{L2}' y='25'>vallée 2-D : sigma (CSA seul, sans restart)</text>");
    sb.Append($"<path d='M{L2} {T2} V{T2 + H2} H{L2 + W2}' fill='none' stroke='black'/>");
    double sMax = eigLog.Max(e => e.Sigma);
    double SP(double v) => T2 + H2 * (1.0 - v / sMax);
    for (int tick = 0; tick <= 2; tick++)
    {
        double v = sMax * tick / 2.0;
        sb.Append($"<text x='{L2 - 40}' y='{F(SP(v) + 4)}'>{v:F2}</text><path d='M{L2 - 4} {F(SP(v))} H{L2}' stroke='black'/>");
    }
    foreach (int tick in new[] { 40, 80, 120 })
    {
        double x = L2 + W2 * tick / 159.0;
        sb.Append($"<text x='{F(x - 10)}' y='{F(T2 + H2 + 18)}'>{tick}</text><path d='M{F(x)} {F(T2 + H2)} v4' stroke='black'/>");
    }
    var epts = string.Join(" ", eigLog.Select(e => $"{F(L2 + W2 * e.Gen / 159.0)} {F(SP(e.Sigma))}"));
    sb.Append($"<polyline fill='none' stroke='#21856b' points='{epts}'/>");
    sb.Append($"<text x='{L2 + W2 / 2 - 40}' y='405'>génération</text>");
    return sb.Append("</svg>").ToString();
}
display(new PlotSvg(CsaFigure()));


Rastrigin d=5 : log10 sigma, redémarrages marqués -3 -2 -1 0 1 100 200 300 <polyline fill='none' stroke='#276fbf' points='65.00 120.42 67.15 116.02 69.30 116.62 71.45 115.74 73.60 106.90 75.75 106.96 77.90 109.18 80.05 111.69 82.20 118.28 84.35 120.34 86.50 117.58 88.65 113.85 90.80 113.69 92.95 116.29 95.10 119.28 97.25 123.52 99.40 121.00 101.55 115.43 103.70 110.29 105.85 103.46 108.00 101.37 110.15 105.03 112.30 109.75 114.45 114.40 116.60 111.92 118.75 109.00 120.90 116.22 123.05 111.70 125.20 117.28 127.35 120.58 129.50 124.40 131.65 119.14 133.80 116.17 135.95 118.84 138.10 117.82 140.25 114.85 142.40 122.80 144.55 118.28 146.70 115.49 148.85 103.12 151.00 107.58 153.15 107.56 155.30 111.26 157.45 118.38 159.60 113.06 161.75 110.47 163.90 107.14 166.05 110.96 168.20 118.08 170.35 116.94 172.50 116.17 174.65 118.97 176.80 119.63 178.95 123.04 181.10 116.82 183.25 116.46 185.40 119.76 187.55 123.44 189.70 112.27 191.85 118.12 194.00 120.83 196.15 126.21 198.30 129.33 200.45 132.24 202.60 133.23 204.75 139.82 206.90 138.41 209.05 130.72 211.20 131.07 213.35 132.75 215.50 130.24 217.65 134.89 219.80 131.83 221.95 123.34 224.10 125.08 226.25 124.03 228.40 121.95 230.55 125.05 232.70 127.32 234.85 126.28 237.00 126.48 239.15 122.55 241.30 124.67 243.45 131.52 245.60 135.78 247.75 135.88 249.90 138.78 252.05 139.57 254.20 130.65 256.35 130.60 258.50 132.86 260.65 139.13 262.80 145.16 264.95 146.26 267.10 148.75 269.25 150.62 271.40 152.64 273.55 153.34 275.70 158.62 277.85 166.27 280.00 168.32 282.15 123.76 284.30 130.95 286.45 126.73 288.60 129.73 290.75 135.72 292.90 128.18 295.05 129.41 297.20 133.75 299.35 136.43 301.50 139.26 303.65 141.77 305.80 145.82 307.95 151.02 310.10 156.41 312.25 160.64 314.40 162.07 316.55 167.94 318.70 174.47 320.85 179.34 323.00 180.86 325.15 190.71 327.30 196.73 329.45 208.37 331.60 217.67 333.75 229.60 335.90 237.91 338.05 243.68 340.20 246.56 342.35 249.95 344.50 254.93 346.65 254.73 348.80 258.00 350.95 259.96 353.10 264.70 355.25 270.72 357.40 273.78 359.55 275.77 361.70 277.03 363.85 283.50 366.00 293.42 368.15 297.31 370.30 303.56 372.45 307.28 374.60 312.03 376.75 313.70 378.90 321.29 381.05 329.00 383.20 338.24 385.35 343.76 387.50 345.00 389.65 345.00 391.80 345.00 393.95 345.00 396.10 345.00 398.25 345.00 400.40 345.00 402.55 345.00 404.70 345.00 406.85 345.00 409.00 345.00 411.15 345.00 413.30 345.00 415.45 345.00 417.60 345.00 419.75 345.00 421.90 345.00 424.05 345.00 426.20 345.00 428.35 345.00 430.50 345.00 432.65 345.00 434.80 345.00 436.95 345.00 439.10 345.00 441.25 345.00 443.40 345.00 445.55 345.00 447.70 345.00 449.85 345.00 452.00 345.00 454.15 345.00 456.30 345.00 458.45 345.00 460.60 345.00 462.75 345.00 464.90 345.00 467.05 345.00 469.20 345.00 471.35 345.00 473.50 345.00 475.65 345.00 477.80 345.00 479.95 345.00 482.10 345.00 484.25 345.00 486.40 345.00 488.55 345.00 490.70 345.00 492.85 345.00'/> sigma | restart-on-stall vallée 2-D : sigma (CSA seul, sans restart) 0,00 1,15 2,30 40 80 120 génération

### Lecture de la figure

À gauche, $\log_{10}\sigma$ sur Rastrigin : décroissance en dents de scie, et à chacune des trois lignes rouges pointillées (redémarrage) le pas saute brutalement vers $10^0$ avant de redécroître — la signature du restart-on-stall. Le plancher d'affichage à $10^{-3}$ masque les $\sigma$ microscopiques atteints en fin de stagnation. À droite, la vallée 2-D sans redémarrage : la même décroissance, mais continue — sur un problème unimodal, l'effondrement de $\sigma$ est la convergence elle-même, pas un piège.


### Exercice 1 : IPOP côté appelant

Le plafond déclaré de la classe se lève **hors** du composé : relire `DefaultPopulationSize(d)`, puis enchaîner des runs complets dont la population double à chaque redémarrage ($\lambda \to 2\lambda$), chaque run repartant du meilleur point du précédent. Sur Rastrigin 5-D avec un budget total d'évaluations fixé (par exemple 12 000), comparer : restart-on-stall à population **constante** (le run ci-dessus) vs IPOP à population doublée — nombre d'évaluations jusqu'à $f \le 10^{-3}$, et nombre de redémarrages (bassins visités).


In [11]:
// Exercice 1 : IPOP cote appelant -- population doublee a chaque redemarrage.
// Squelette d'accompagnement : la boucle reste a ecrire.
// Etape 1 : budget total (p. ex. 12000 evaluations) et lambda0 = C.DefaultPopulationSize(dR).
// Etape 2 : enchaîner des runs ; centrer chaque depart sur le BestPoint du run precedent
//           (mean initiale de CmaEsDistribution.Create, ou population MetaPopulation centree).
// Etape 3 : a chaque stall, lambdaK *= 2 puis nouveau run ; cumuler les evaluations.
// Etape 4 : mesurer l'evaluation ou f <= 1e-3 tombe, et le nombre de redemarrages.
Console.WriteLine("Exercice a completer : IPOP cote appelant (population doublee a chaque redemarrage)");
Console.WriteLine("  1. lambda0 = C.DefaultPopulationSize(5) ; budget total 12000 evaluations");
Console.WriteLine("  2. enchaîner les runs, centrer chaque depart sur le BestPoint precedent");
Console.WriteLine("  3. comparer a population constante : evals jusqu'a f <= 1e-3, bassins visites");
object ipopResult = null; // TODO etudiant : (evals, restarts) du run IPOP
// Indice : Restart(bestPoint, stepSize) du compose ne touche pas la population ;
// la taille de population ne se change qu'en reconstruisant la MetaPopulation du run suivant.


Exercice a completer : IPOP cote appelant (population doublee a chaque redemarrage)


  1. lambda0 = C.DefaultPopulationSize(5) ; budget total 12000 evaluations


  2. enchaîner les runs, centrer chaque depart sur le BestPoint precedent


  3. comparer a population constante : evals jusqu'a f <= 1e-3, bassins visites


### Exercice 2 : le prix du conditionnement

Le banc fixe le rapport d'échelles à $10^4$. Faire croître le conditionnement — $\{10^2, 10^3, 10^4, 10^5, 10^6\}$ en gardant la même rotation — et tracer, pour CMA-ES puis xNES (3 graines, médiane), le nombre d'évaluations jusqu'à $10^{-8}$ (ou le meilleur final si la cible manque). Hypothèse à confronter : l'écart entre les deux croît avec le conditionnement, parce qu'une vallée toujours plus étroite exige la corrélation entre générations — le rang un — que le seul rang-μ n'a pas.


In [12]:
// Exercice 2 : evals jusqu'a 1e-8 en fonction du conditionnement, CMA-ES vs xNES.
// Squelette d'accompagnement -- les runs restent a faire.
// Etape 1 : pour chaque rapport r in {1e2, 1e3, 1e4, 1e5, 1e6}, echelles geometriques
//           ScalesCond[i] = Math.Pow(r, 1.0 - (double)i / (d - 1)) -- meme rotation Q5.
// Etape 2 : parametrer RunGa par l'echelle (cloner le helper avec ScalesCond en argument).
// Etape 3 : 3 graines {4242, 1717, 2718}, mediane des ReachedAt (ou des Best) ; trace ou tableau.
Console.WriteLine("Exercice a completer : prix du conditionnement (1e2 -> 1e6)");
Console.WriteLine("  1. echelles geometriques de rapport r, meme rotation Q5");
Console.WriteLine("  2. CMA-ES vs xNES, 3 graines {4242,1717,2718}, mediane");
Console.WriteLine("  3. l'etirement d'axes de la covariance suit-il sqrt(r) en fin de run ?");
object condResult = null; // TODO etudiant : liste (rapport, evalsCma, evalsXnes)
// Indice : seule la liste des echelles change ; le canevas GA reste identique.


Exercice a completer : prix du conditionnement (1e2 -> 1e6)


  1. echelles geometriques de rapport r, meme rotation Q5


  2. CMA-ES vs xNES, 3 graines {4242,1717,2718}, mediane


  3. l'etirement d'axes de la covariance suit-il sqrt(r) en fin de run ?


### Exercice 3 : invariance par rotation

L'invariance par rotation est la propriété structurante de CMA-ES (test `Update_CommutesWithRotations` : faire tourner l'état et la population, mettre à jour, donne l'état tourné de la mise à jour, écart $< 10^{-10}$). La voir au niveau du **run** : remplacer $Q_5$ par une autre rotation $Q' = $ `Givens(new[] { -0.4, 0.8, 1.3, 0.5 })` et relancer la démo discriminative. Si la propriété se propage aux trajectoires, la courbe CMA-ES doit rester au même ordre — à graine égale, l'écart mesure ce que le pipeline GA ajoute autour de la distribution.


In [13]:
// Exercice 3 : invariance par rotation au niveau du run.
// Squelette d'accompagnement -- le run reste a faire.
// Etape 1 : Q5b = Givens(new[] { -0.4, 0.8, 1.3, 0.5 }) ; controle d'orthogonalite Q5b^T Q5b = I.
// Etape 2 : relancer le canevas (parametre de rotation) sur CMA-ES, graine 4242.
// Etape 3 : comparer ReachedAt / Best au run de reference ; meme controle sur xNES.
Console.WriteLine("Exercice a completer : invariance par rotation (rotation alternative Q')");
Console.WriteLine("  1. Q' = Givens({ -0.4, 0.8, 1.3, 0.5 }) ; controle Q'^T Q' = I");
Console.WriteLine("  2. CMA-ES graine 4242 sur Q' : ReachedAt et Best attendus proches du run Q5");
Console.WriteLine("  3. meme controle sur xNES : l'invariance est-elle le propre de CMA-ES ?");
object rotResult = null; // TODO etudiant : (reachedAtPrime, bestPrime) sur Q'
// Indice : le test unitaire compare des ETATS (l'update commute avec la rotation) ;
// ici on compare des RUNS -- l'eventuel echec de SNES est le sujet meme de la vallee tournee.


Exercice a completer : invariance par rotation (rotation alternative Q')


  1. Q' = Givens({ -0.4, 0.8, 1.3, 0.5 }) ; controle Q'^T Q' = I


  2. CMA-ES graine 4242 sur Q' : ReachedAt et Best attendus proches du run Q5


  3. meme controle sur xNES : l'invariance est-elle le propre de CMA-ES ?


## Liens

- [Search-05b NES en C#](../Part1-Foundations/Search-05b-NaturalEvolutionStrategies-CSharp.ipynb) — le lien au premier ordre xNES ↔ rang-μ, et la référence Python `cma` (ellipsoïde tourné : 3 850 évaluations jusqu'à $10^{-8}$).
- [MGS-07d MichalewiczDixonPrice](MGS-07d-MichalewiczDixonPrice.ipynb) — l'exercice 3 (anisotropie V/H) dont ce carnet donne la réponse algébrique.
- Issue [#24012](https://github.com/jsboige/CoursIA/issues/24012) — onboarding CMA-ES natif (le chaînon manquant, la clause « Sans objet », les phases).
- PR [#24035](https://github.com/jsboige/CoursIA/pull/24035) et [jsboige/MetaGeneticSharp#71](https://github.com/jsboige/MetaGeneticSharp/pull/71) — la phase 1 : classe + tests.
- [jsboige/MetaGeneticSharp#75](https://github.com/jsboige/MetaGeneticSharp/pull/75) — l'orchestrateur de redémarrages à populations croissantes (tranche partielle).
- Tutoriel **N. Hansen, *The CMA Evolution Strategy: A Tutorial*** ([arXiv 1604.00772](https://arxiv.org/abs/1604.00772), publié dans *Evolutionary Computation* 2024).


## À retenir

1. **Le cadre porte l'écart, le rang-un le raffine.** Ellipsoïde 5-D tourné, conditionnement $10^4$, même population et mêmes opérateurs pour tous : CMA-ES atteint $10^{-8}$ sur 3 graines sur 3 en **2 271 évaluations** (médiane), meilleur $1{,}97\times10^{-16}$. L'ablation du seul $c_1$ — CSA, redémarrage, budget et initialisation inchangés — garde **3/3** la cible à $2{,}33\times10^{-14}$ en 2 611 évaluations : ~2 ordres de grandeur et ~15 % de budget, cohérents sur les trois graines. xNES ($3{,}45\times10^{-4}$), SNES ($1{,}18\times10^{-1}$) et DE ($1{,}70\times10^{-2}$) restent hors cible : l'écart de douze ordres est porté par le cadre complet, pas par le repli $p_c p_c^\top$ seul.
2. **La passe de référence reproduit le test à l'évaluation près** : 2 269 évaluations, meilleur $1{,}97\times10^{-16}$, graine 4242 — la transcription du banc est fidèle au témoin unitaire.
3. **La covariance apprend la géométrie.** En 2-D, l'angle de l'axe principal se cale sur l'orientation théorique de la vallée (≈39° mesurés contre 38,4° calculés) dès la génération ~25 ; le rapport d'axes croise sa cible $\sqrt{25}=5$ vers la génération 35, puis l'apprentissage gèle avec l'effondrement de $\sigma$ à la convergence ($f \sim 10^{-28}$).
4. **Le CSA suit la trajectoire, le restart la relance.** $\sigma$ décroît de 2,30 à ~0 en ~50 générations sur la vallée unimodale ; sur Rastrigin 5-D, 3 redémarrages (stall = 40 générations) ré-injectent chacun $\sigma=1$ autour du meilleur point, qui passe de $f \approx 80$ à **5,97**.
5. **Le plafond IPOP est déclaré, pas contourné** : la population appartient au GA hôte, pas au composé — la classe le dit verbatim, l'orchestrateur dédié existe à part (MetaGeneticSharp#75), et l'exercice 1 construit la boucle $\lambda \to 2\lambda$ côté appelant.

## Vérifiez votre compréhension

1. **Pourquoi SNES finit-il derrière DE, qui n'a pourtant aucun modèle de distribution ?**
2. **Le rapport d'axes dépasse sa cible (jusqu'à 8,79 vers la génération 85) puis redescend — que mesure-t-on vraiment après l'effondrement de $\sigma$ ?**
3. **Pourquoi le redémarrage ne peut-il pas doubler la population depuis l'intérieur du composé ?**
4. **En quoi le CSA se distingue-t-il d'un $\sigma$ décroissant programmé à l'avance ?**
5. **L'ablation du rang-un garde la cible sur 3 graines à ~2 ordres de grandeur du bras complet — pourquoi ce résultat réattribue-t-il l'écart massif au lieu de le confirmer ?**

<details>
<summary>Réponses</summary>

1. Une covariance diagonale apprend des échelles par **axe de coordonnées**, alors que la vallée tournée vit selon ses propres axes : le modèle contraint la recherche dans la direction fausse (mesuré : $1{,}2\times10^{-1}$). DE, sans modèle, forme ses directions de croisement par différences de vecteurs courants — qui suivent librement la pente locale ($1{,}7\times10^{-2}$). Un modèle faux est plus coûteux que pas de modèle.
2. Plus grand-chose : les mises à jour de $C$ dépendent des écarts échantillonnés, proportionnels à $\sigma$ ; une fois $\sigma$ effondré, ces écarts ne portent plus de signal géométrique et le rapport mesuré dérive par bruit numérique. Le pic de 8,79 n'est plus de l'apprentissage — c'est l'affichage d'un état figé par la convergence.
3. La taille de population est un paramètre du `MetaGeneticAlgorithm` hôte, fixé à la construction ; le composé ne pilote que la distribution. Doubler $\lambda$ en cours de run demanderait de reconstruire la population — un pouvoir que la classe déclare explicitement hors de sa portée (remarques verbatim) et que l'exercice 1 exerce côté appelant.
4. Le CSA compare la longueur du chemin whitened $C^{-1/2}p_\sigma$ à son espérance $\mathbb{E}\|\mathcal{N}(0,I)\|$ : $\sigma$ **monte** si la moyenne progresse de façon régulière (chemin long = direction exploitable), descend sinon. Un decay programmé ignore la trajectoire : sur Rastrigin, il ne pourrait pas distinguer la convergence (resserrer) de la stagnation (repartir) — le CSA, lui, alimente le détecteur de stall.
5. Parce que l'expérience est **appariée** : les deux bras ne diffèrent que par $c_1$. Si le rang-un portait l'écart de douze ordres de grandeur, le neutraliser aurait dû ramener CMA-ES vers le plateau de xNES ($10^{-4}$) ; il reste à $10^{-14}$ avec la cible atteinte sur les trois graines. CMA-ES a le rang-un **et** les douze ordres — corrélation ; l'ablation isole le terme, mesure ~2 ordres, et montre que ce n'est pas lui le porteur : causalité. L'écart massif vit dans ce que l'ablation n'a pas touché (CSA, redémarrage, paramétrisation de la mise à jour) et demanderait d'autres ablations pour être départagé.

</details>

## Pour aller plus loin

- **Carnet suivant** : [MGS-8 LandscapeExplorer](MGS-08-LandscapeExplorer.ipynb) — l'explorateur interactif de paysages.
- **Dans la même série** : [MGS-07d MichalewiczDixonPrice](MGS-07d-MichalewiczDixonPrice.ipynb) — les crêtes raides et la vallée non-scalable dont celui-ci apprend la géométrie ; [MGS-13 LandscapeDebias](MGS-13-LandscapeDebias.ipynb) — quand la rotation des axes change le verdict d'un algorithme.
- **Point d'entrée** : [README de la série](README.md) — positionnement MGS face à GeneticSharp et mealpy.
- **Dans le code** : `Metaheuristics/Compound/CovarianceMatrixAdaptationEvolutionStrategy.cs` du [fork jsboige/MetaGeneticSharp](https://github.com/jsboige/MetaGeneticSharp) — rang un (éq. 43), CSA (éq. 44-46), `CmaEsConstants` (table 1 du tutoriel) — et son test `CovarianceMatrixAdaptationEvolutionStrategyTests.cs`.
